In [1]:
import open3d as o3d 
import numpy as np

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
def get_alignment_matrix(source_path, target_path):
    source = o3d.io.read_point_cloud(source_path) # GS Map
    target = o3d.io.read_point_cloud(target_path) # GT Map

    # Initial guess (identity or manual offset)
    trans_init = np.identity(4)

    # Run ICP
    reg_p2p = o3d.pipelines.registration.registration_icp(
        source, target, max_correspondence_distance=0.05,
        init=trans_init,
        estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint()
    )
    return reg_p2p

In [4]:
gt_map = "/data/online_lang_splatting/data/vmap/room_0/mesh.ply"
gs_map = "/data/online_lang_splatting/output/room_0/omni_data_result/room_0/2026-05-07-10-05-19/point_cloud/final/point_cloud.ply"
alignment_matrix = get_alignment_matrix(gs_map, gt_map)
print("Alignment Matrix:")
print(alignment_matrix.transformation)

Alignment Matrix:
[[ 0.99995763  0.00893056  0.00223103 -0.01495428]
 [-0.00897161  0.99977685  0.01912481 -0.01209679]
 [-0.00205974 -0.01914401  0.99981461  0.03072428]
 [ 0.          0.          0.          1.        ]]


In [7]:

def apply_transform_to_gs(ply_path, matrix, output_path):
    plydata = PlyData.read(ply_path)
    v = plydata['vertex']
    
    # Extract Rotation and Translation from the 4x4 matrix
    R_mat = matrix[:3, :3]
    t_vec = matrix[:3, 3]
    
    # 1. Transform Positions
    xyz = np.stack((v['x'], v['y'], v['z']), axis=-1)
    new_xyz = (R_mat @ xyz.T).T + t_vec
    
    # 2. Transform Quaternions (Rotations)
    # 3DGS Quats are usually [rot_0, rot_1, rot_2, rot_3] as [w, x, y, z]
    raw_quats = np.stack((v['rot_1'], v['rot_2'], v['rot_3'], v['rot_0']), axis=-1) # to [x,y,z,w]
    
    r_correction = R.from_matrix(R_mat)
    # Apply rotation to every individual Gaussian's orientation
    old_rots = R.from_quat(raw_quats)
    new_rots = r_correction * old_rots
    new_quats = new_rots.as_quat() # [x, y, z, w]

    # 3. Reconstruct and Save
    # Create new vertex data preserving all other attributes (SHs, opacity, scale)
    new_data = np.array(v.data, copy=True)
    new_data['x'], new_data['y'], new_data['z'] = new_xyz[:,0], new_xyz[:,1], new_xyz[:,2]
    new_data['rot_0'] = new_quats[:,3] # w
    new_data['rot_1'] = new_quats[:,0] # x
    new_data['rot_2'] = new_quats[:,1] # y
    new_data['rot_3'] = new_quats[:,2] # z
    
    PlyData([PlyElement.describe(new_data, 'vertex')]).write(output_path)

# matrix = (The 4x4 matrix you got from ICP)
# apply_transform_to_gs("wrong_gs.ply", matrix, "correct_gs.ply")

In [8]:
apply_transform_to_gs(gs_map, alignment_matrix, "/data/online_lang_splatting/output/room_0/omni_data_result/room_0/2026-05-07-10-05-19/point_cloud/final/corrected_point_cloud.ply")


In [5]:
import open3d as o3d
import numpy as np
import copy

def get_relative_transform(source_pc, target_pc, threshold=0.02):
    # 1. Initial alignment (Identity matrix assumes they are roughly aligned)
    # If they are far apart, you'll need a global registration first.
    trans_init = np.identity(4)

    # 2. Run Point-to-Point ICP
    reg_p2p = o3d.pipelines.registration.registration_icp(
        source_pc, target_pc, threshold, trans_init,
        o3d.pipelines.registration.TransformationEstimationPointToPoint())

    # This is the 4x4 Matrix representing Map B's origin/rotation in Map A's frame
    transformation = reg_p2p.transformation
    
    # 3. Extract components
    rotation_matrix = transformation[:3, :3]
    origin_offset = transformation[:3, 3] # This is the [x, y, z] relative origin
    
    return transformation, rotation_matrix, origin_offset


In [6]:

# Usage
source = o3d.io.read_point_cloud(gs_map) # The map you want to move
target = o3d.io.read_point_cloud(gt_map) # The fixed reference map

T, R, t = get_relative_transform(source, target)

print("Origin of Map B relative to Map A (x, y, z):")
print(t)
print("\nRotation Matrix:")
print(R)

Origin of Map B relative to Map A (x, y, z):
[-0.00285147 -0.00248994  0.01034618]

Rotation Matrix:
[[ 9.99994915e-01  3.18735550e-03 -1.07821949e-04]
 [-3.18682608e-03  9.99984352e-01  4.59782701e-03]
 [ 1.22475171e-04 -4.59746002e-03  9.99989424e-01]]
